# 🏃 Running Coach Agent Swarm
## Tool Layer Test Suite

Tests all four tools in the `tools/` directory:
1. `parse_workout_data` — load, retrieve, validate, summarise WorkoutRecords
2. `calculate_pace_zones` — workout classification and pace evaluation
3. `calculate_training_load` — ATL, CTL, TSB, mileage rule, weekly summaries
4. `query_garmin_db` — HRV, sleep, body battery, stress, readiness, race predictions

Each section prints a clear ✅ / ❌ / ⚠️ result so you can see at a glance what's working.

---
## Setup — Mount Drive & Import Tools

In [1]:
import os, sys, sqlite3
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE_DIR        = "/content/drive/MyDrive/running_coach"
WORKOUTS_PATH   = f"{BASE_DIR}/data/processed/workouts_normalized.json"
GARMIN_DB_PATH  = f"{BASE_DIR}/data/raw/garmin/garmin.db"

sys.path.insert(0, f"{BASE_DIR}/tools")

from parse_workout_data   import (
    load_workouts, get_workout_by_id, get_workouts_by_date,
    get_recent_workouts, get_workouts_by_type,
    validate_all, summarise_workouts, format_pace
)
from calculate_pace_zones import (
    classify_workouts, detect_workout_type,
    evaluate_pace, resolve_workout_type, format_pace as fmt_pace
)
from calculate_training_load import (
    get_current_metrics, get_recent_trend,
    check_recovery_alert, weekly_load_summary,
    check_mileage_rule, format_metrics_report
)
from query_garmin_db import (
    get_hrv, get_sleep, get_body_battery, get_stress,
    get_training_readiness, get_resting_hr, get_race_predictions,
    get_recent_snapshots, format_recovery_context, format_race_predictions
)

print("✅ Drive mounted and all tools imported.")
print(f"   Workouts file exists: {os.path.exists(WORKOUTS_PATH)}")
print(f"   Garmin DB exists:     {os.path.exists(GARMIN_DB_PATH)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted and all tools imported.
   Workouts file exists: True
   Garmin DB exists:     True


---
## Tool 1 — `parse_workout_data`

In [2]:
print("=" * 60)
print("TEST 1.1 — Load workouts (180 days)")
print("=" * 60)
try:
    workouts = load_workouts(WORKOUTS_PATH, days=180)
    assert len(workouts) > 0, "No workouts loaded"
    print(f"✅ Loaded {len(workouts)} workouts")
    print(f"   Date range: {workouts[-1]['date']} → {workouts[0]['date']}")
    print(f"   Sources:    {set(w['source'] for w in workouts)}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 1.1 — Load workouts (180 days)
✅ Loaded 115 workouts
   Date range: 2025-11-05 → 2026-05-02
   Sources:    {'strava'}


In [3]:
print("=" * 60)
print("TEST 1.2 — Retrieval helpers")
print("=" * 60)

# get_recent_workouts
try:
    recent = get_recent_workouts(workouts, n=5)
    assert len(recent) == 5
    print(f"✅ get_recent_workouts(5): {[w['date'] for w in recent]}")
except Exception as e:
    print(f"❌ get_recent_workouts: {e}")

# get_workouts_by_date
try:
    test_date = workouts[0]['date']
    on_date = get_workouts_by_date(workouts, test_date)
    print(f"✅ get_workouts_by_date('{test_date}'): {len(on_date)} record(s)")
except Exception as e:
    print(f"❌ get_workouts_by_date: {e}")

# get_workout_by_id
try:
    test_id = workouts[0]['activity_id']
    found = get_workout_by_id(workouts, test_id)
    assert found is not None
    print(f"✅ get_workout_by_id('{test_id}'): found {found['date']}")
except Exception as e:
    print(f"❌ get_workout_by_id: {e}")

# get_workouts_by_type
try:
    halifax = get_workouts_by_type(workouts, 'Halifax')
    print(f"✅ get_workouts_by_type('Halifax'): {len(halifax)} record(s)")
except Exception as e:
    print(f"❌ get_workouts_by_type: {e}")

TEST 1.2 — Retrieval helpers
✅ get_recent_workouts(5): ['2026-05-02', '2026-04-30', '2026-04-29', '2026-04-28', '2026-04-26']
✅ get_workouts_by_date('2026-05-02'): 1 record(s)
✅ get_workout_by_id('strava_18345065479'): found 2026-05-02
✅ get_workouts_by_type('Halifax'): 0 record(s)


In [4]:
print("=" * 60)
print("TEST 1.3 — Schema validation")
print("=" * 60)
try:
    results = validate_all(workouts, print_results=True)
    invalid = [r for r in results if not r.is_valid]
    if not invalid:
        print("✅ All records passed validation")
    else:
        print(f"⚠️  {len(invalid)} invalid records — see above")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 1.3 — Schema validation
Validation summary — 115 records
  ✅ Clean:    113
  ⚠️  Warnings: 2
  ❌ Invalid:  0

Records with warnings:
✅ valid | strava_18171266115 | 2026-04-11
✅ valid | strava_17730469106 | 2026-03-15
✅ All records passed validation


In [5]:
print("=" * 60)
print("TEST 1.4 — Summary statistics")
print("=" * 60)
try:
    summary = summarise_workouts(workouts)
    assert summary.get('count', 0) > 0
    print(f"✅ summarise_workouts:")
    for k, v in summary.items():
        print(f"   {k:<28} {v}")
except Exception as e:
    print(f"❌ FAILED: {e}")

print("=" * 60)
print("TEST 1.5 — format_pace utility")
print("=" * 60)
test_paces = [4 + 14/60, 4 + 1/60, 3 + 56/60, 3 + 40/60, None]
expected   = ['4:14/km', '4:01/km', '3:56/km', '3:40/km', '--:--/km']
all_ok = True
for pace, exp in zip(test_paces, expected):
    result = format_pace(pace)
    status = '✅' if result == exp else '❌'
    if result != exp:
        all_ok = False
    print(f"  {status} format_pace({pace}) = '{result}'  (expected '{exp}')")
if all_ok:
    print("\n✅ All pace formatting tests passed")

TEST 1.4 — Summary statistics
✅ summarise_workouts:
   count                        115
   total_distance_km            1262.41
   avg_distance_km              10.98
   avg_pace_min_km              4.93
   avg_pace_formatted           4:56/km
   avg_hr                       138.9
   avg_suffer_score             49.28
   avg_training_load            141.26
   avg_aerobic_effect           3.37
   garmin_enriched_count        114
   date_range_start             2025-11-05
   date_range_end               2026-05-02
TEST 1.5 — format_pace utility
  ✅ format_pace(4.233333333333333) = '4:14/km'  (expected '4:14/km')
  ✅ format_pace(4.016666666666667) = '4:01/km'  (expected '4:01/km')
  ✅ format_pace(3.9333333333333336) = '3:56/km'  (expected '3:56/km')
  ✅ format_pace(3.6666666666666665) = '3:40/km'  (expected '3:40/km')
  ✅ format_pace(None) = '--:--/km'  (expected '--:--/km')

✅ All pace formatting tests passed


---
## Tool 2 — `calculate_pace_zones`

In [6]:
print("=" * 60)
print("TEST 2.1 — Workout classification")
print("=" * 60)
try:
    enriched     = classify_workouts(workouts)
    structured   = [w for w in enriched if w['classification']['is_structured']]
    easy         = [w for w in enriched if w['classification']['workout_type'] == 'easy']
    needs_input  = [w for w in enriched if w['classification']['needs_input']]
    exceptions   = [w for w in enriched if w['classification']['detected_from'] == 'known_exception']

    print(f"✅ classify_workouts({len(workouts)} records):")
    print(f"   Structured (Garmin date-named): {len(structured)}")
    print(f"   Easy (auto-classified):         {len(easy)}")
    print(f"   Known exceptions:               {len(exceptions)}")
    print(f"   Needs user input:               {len(needs_input)}")

    if needs_input:
        print(f"\n   ⚠️  Unclassified activities (review these):")
        for w in needs_input:
            print(f"   {w['date']} | garmin: '{w.get('garmin_name','')}' | strava: '{w.get('name','')}' | {fmt_pace(w.get('avg_pace_min_km'))}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 2.1 — Workout classification
✅ classify_workouts(115 records):
   Structured (Garmin date-named): 35
   Easy (auto-classified):         78
   Known exceptions:               1
   Needs user input:               2

   ⚠️  Unclassified activities (review these):
   2025-12-20 | garmin: 'Halifax - Run Workout (2)' | strava: 'Morning Run' | 4:23/km
   2025-11-08 | garmin: 'Indianapolis - Indy' | strava: 'Morning Run' | 4:41/km


In [7]:
print("=" * 60)
print("TEST 2.2 — Easy run pace evaluation (sample of 5)")
print("=" * 60)
try:
    easy_runs = [w for w in enriched if w['classification']['workout_type'] == 'easy'][:5]
    for w in easy_runs:
        pe = w.get('pace_evaluation')
        if pe:
            print(f"  {w['date']} | {pe['actual_pace_fmt']} | {pe['verdict']}")
        else:
            print(f"  {w['date']} | no evaluation")
    print("✅ Easy run evaluation complete")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 2.2 — Easy run pace evaluation (sample of 5)
  2026-04-30 | 5:08/km | Easy run pace of 5:08/km is within easy territory (above 4:45/km ceiling). ✅
  2026-04-28 | 5:01/km | Easy run pace of 5:01/km is within easy territory (above 4:45/km ceiling). ✅
  2026-04-26 | 5:17/km | Easy run pace of 5:17/km is within easy territory (above 4:45/km ceiling). ✅
  2026-04-24 | 5:55/km | Easy run pace of 5:55/km is within easy territory (above 4:45/km ceiling). ✅
  2026-04-23 | 5:05/km | Easy run pace of 5:05/km is within easy territory (above 4:45/km ceiling). ✅
✅ Easy run evaluation complete


In [8]:
print("=" * 60)
print("TEST 2.3 — Structured workout pace evaluation (sample of 5)")
print("=" * 60)
try:
    structured_sample = [w for w in enriched if w['classification']['is_structured']][:5]
    for w in structured_sample:
        cl = w['classification']
        pe = w.get('pace_evaluation')
        print(f"  {w['date']} | type: {cl['workout_type'] or 'pending'} | from: {cl['detected_from']}")
        if pe:
            print(f"    → {pe['verdict']}")
        else:
            print(f"    → {cl['notes']}")
    print("\n✅ Structured workout evaluation complete")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 2.3 — Structured workout pace evaluation (sample of 5)
  2026-05-02 | type: pending | from: garmin_name
    → Cannot evaluate — target pace or actual pace is missing.
  2026-04-29 | type: pending | from: garmin_name
    → Cannot evaluate — target pace or actual pace is missing.
  2026-04-25 | type: pending | from: garmin_name
    → Cannot evaluate — target pace or actual pace is missing.
  2026-04-22 | type: pending | from: garmin_name
    → Cannot evaluate — target pace or actual pace is missing.
  2026-04-18 | type: pending | from: garmin_name
    → Cannot evaluate — target pace or actual pace is missing.

✅ Structured workout evaluation complete


In [9]:
print("=" * 60)
print("TEST 2.4 — resolve_workout_type (simulates Feedback Agent user input)")
print("=" * 60)
try:
    # Simulate classifying an unknown workout and resolving its type
    mock_workout = {
        'activity_id': 'strava_test_001',
        'date': '2026-04-10',
        'name': 'Morning Run',
        'garmin_name': '',
        'avg_pace_min_km': 4 + 3/60,   # 4:03/km — close to threshold
    }
    classification = detect_workout_type(mock_workout)
    print(f"  Initial classification: needs_input={classification.needs_input}, "
          f"detected_from='{classification.detected_from}'")

    resolved = resolve_workout_type(classification, 'threshold')
    print(f"  After resolve:          workout_type='{resolved.workout_type}', "
          f"needs_input={resolved.needs_input}")

    evaluation = evaluate_pace(mock_workout, resolved)
    print(f"  Pace evaluation: {evaluation.verdict}")
    print("✅ resolve_workout_type working correctly")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 2.4 — resolve_workout_type (simulates Feedback Agent user input)
  Initial classification: needs_input=False, detected_from='name_pattern'
  After resolve:          workout_type='threshold', needs_input=False
  Pace evaluation: Threshold target: 4:01/km. Actual: 4:03/km (2s slower than target). On target. ✅
✅ resolve_workout_type working correctly


---
## Tool 3 — `calculate_training_load`

In [10]:
print("=" * 60)
print("TEST 3.1 — Current ATL/CTL/TSB metrics")
print("=" * 60)
try:
    metrics = get_current_metrics(workouts)
    assert metrics is not None
    print(f"✅ get_current_metrics:")
    print(f"   Date:          {metrics.date}")
    print(f"   ATL (fatigue): {metrics.atl}")
    print(f"   CTL (fitness): {metrics.ctl}")
    print(f"   TSB (form):    {metrics.tsb}  → {metrics.form_label}")
    print(f"   Fitness level: {metrics.fitness_label}")
    print(f"   Alert:         {metrics.in_alert}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.1 — Current ATL/CTL/TSB metrics
  ℹ️  1 record(s) skipped (no training_load — not Garmin-enriched). Treated as 0 load.
✅ get_current_metrics:
   Date:          2026-05-03
   ATL (fatigue): 82.67
   CTL (fitness): 99.96
   TSB (form):    17.29  → Very fresh — peak form
   Fitness level: High fitness
   Alert:         False


In [11]:
print("=" * 60)
print("TEST 3.2 — 14-day trend")
print("=" * 60)
try:
    trend = get_recent_trend(workouts, days=14)
    assert trend is not None
    print(f"✅ get_recent_trend(14):")
    print(f"   ATL:  {trend.atl_start:.1f} → {trend.atl_end:.1f}  ({trend.atl_trend})")
    print(f"   CTL:  {trend.ctl_start:.1f} → {trend.ctl_end:.1f}  ({trend.ctl_trend})")
    print(f"   TSB:  {trend.tsb_start:.1f} → {trend.tsb_end:.1f}  ({trend.tsb_trend})")
    print(f"   Total load:     {trend.total_load}")
    print(f"   Avg daily load: {trend.avg_daily_load}")
    print(f"   Peak load:      {trend.peak_load}")
    print(f"   Rest days:      {trend.rest_days} / {trend.days}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.2 — 14-day trend
✅ get_recent_trend(14):
   ATL:  107.1 → 82.7  (falling)
   CTL:  106.4 → 100.0  (falling)
   TSB:  -0.7 → 17.3  (rising)
   Total load:     1081.3
   Avg daily load: 77.2
   Peak load:      229.0
   Rest days:      4 / 14


In [12]:
print("=" * 60)
print("TEST 3.3 — Recovery alert check")
print("=" * 60)
try:
    alert = check_recovery_alert(workouts)
    if alert:
        print(f"⚠️  Recovery alert triggered:")
        print(f"   {alert.message}")
    else:
        print(f"✅ No recovery alert — TSB within acceptable range")
        print(f"   Current TSB: {metrics.tsb:.1f}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.3 — Recovery alert check
✅ No recovery alert — TSB within acceptable range
   Current TSB: 17.3


In [13]:
print("=" * 60)
print("TEST 3.4 — Mileage rule check")
print("=" * 60)
try:
    mileage = check_mileage_rule(workouts)
    status = '✅' if mileage['ok'] else '⚠️'
    print(f"{status} check_mileage_rule:")
    print(f"   {mileage['message']}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.4 — Mileage rule check
✅ check_mileage_rule:
   ✅ Weekly load on track: -21.9% vs last week (607). Within 10% rule.


In [14]:
print("=" * 60)
print("TEST 3.5 — Weekly load summary (8 weeks)")
print("=" * 60)
try:
    weeks = weekly_load_summary(workouts, weeks=8)
    assert len(weeks) > 0
    print(f"✅ weekly_load_summary — {len(weeks)} weeks:")
    print(f"  {'Week':<12} {'Load':>8} {'Sessions':>9} {'Rest':>5} "
          f"{'ATL':>6} {'CTL':>6} {'TSB':>6} {'Change':>8}")
    print("  " + "-" * 62)
    for w in weeks:
        change = f"{w.pct_change_load:+.1f}%" if w.pct_change_load is not None else "  base"
        print(
            f"  {w.week_start:<12} {w.total_load:>8.1f} {w.session_count:>9} "
            f"{w.rest_days:>5} {w.avg_atl:>6.1f} {w.avg_ctl:>6.1f} "
            f"{w.avg_tsb:>6.1f} {change:>8}"
        )
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.5 — Weekly load summary (8 weeks)
✅ weekly_load_summary — 8 weeks:
  Week             Load  Sessions  Rest    ATL    CTL    TSB   Change
  --------------------------------------------------------------
  2026-04-27      474.1         4     3   87.5  101.8   14.3     base
  2026-04-20      607.1         6     1  105.0  106.0    1.0   +28.1%
  2026-04-13      685.7         5     2  123.5  107.9  -15.6   +12.9%
  2026-04-06     1075.4         4     3  146.5  106.5  -40.0   +56.8%
  2026-03-30      990.6         5     2  121.1   97.1  -24.0    -7.9%
  2026-03-23      934.6         5     2   98.6   90.4   -8.2    -5.7%
  2026-03-16      455.2         4     3   89.2   89.0   -0.2   -51.3%
  2026-03-09      770.6         5     2  105.0   89.8  -15.2   +69.3%


In [15]:
print("=" * 60)
print("TEST 3.6 — Full agent-ready metrics report")
print("=" * 60)
try:
    report = format_metrics_report(metrics, trend, alert, mileage)
    print(report)
    print("\n✅ format_metrics_report produced output")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 3.6 — Full agent-ready metrics report
=== Training Load Report — 2026-05-03 ===
  ATL (fatigue):   82.7
  CTL (fitness):   100.0
  TSB (form):      17.3  →  Very fresh — peak form
  Fitness level:   High fitness

  14-day trend:
    ATL:  107.1 → 82.7  (falling)
    CTL:  106.4 → 100.0  (falling)
    TSB:  -0.7 → 17.3  (rising)
    Total load:     1081
    Avg daily load: 77.2
    Peak load:      229.0
    Rest days:      4 / 14

  Mileage rule: ✅ Weekly load on track: -21.9% vs last week (607). Within 10% rule.

✅ format_metrics_report produced output


---
## Tool 4 — `query_garmin_db`

In [16]:
print("=" * 60)
print("TEST 4.0 — Connect to garmin.db")
print("=" * 60)
try:
    conn = sqlite3.connect(GARMIN_DB_PATH)
    row = conn.execute("SELECT COUNT(*) FROM activity").fetchone()
    print(f"✅ Connected — {row[0]} activity records found")
except Exception as e:
    print(f"❌ FAILED to connect: {e}")
    raise

TEST 4.0 — Connect to garmin.db
✅ Connected — 813 activity records found


In [17]:
print("=" * 60)
print("TEST 4.1 — HRV (last 7 days)")
print("=" * 60)
try:
    hrv_records = get_hrv(conn, days=7)
    print(f"✅ get_hrv(7): {len(hrv_records)} records")
    print(f"  {'Date':<12} {'Last Night':>11} {'Weekly Avg':>11} {'Status':<15} {'In Baseline'}")
    print("  " + "-" * 58)
    for h in hrv_records:
        wb = 'Yes ✅' if h.within_baseline is True else \
             'No ⚠️' if h.within_baseline is False else 'N/A'
        print(f"  {h.date:<12} {str(h.last_night):>11} {str(h.weekly_avg):>11} "
              f"{h.status_label:<15} {wb}")

    nulls = sum(1 for h in hrv_records if h.last_night is None)
    if nulls > 0:
        print(f"  ⚠️  {nulls} record(s) with null last_night — check raw_json parsing")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.1 — HRV (last 7 days)
✅ get_hrv(7): 7 records
  Date          Last Night  Weekly Avg Status          In Baseline
  ----------------------------------------------------------
  2026-05-03          None       100.0 Balanced ✅      N/A
  2026-05-02          None        97.0 Balanced ✅      N/A
  2026-05-01          None        92.0 Balanced ✅      N/A
  2026-04-30          None        94.0 Balanced ✅      N/A
  2026-04-29          None        95.0 Balanced ✅      N/A
  2026-04-28          None        95.0 Balanced ✅      N/A
  2026-04-27          None        93.0 Balanced ✅      N/A
  ⚠️  7 record(s) with null last_night — check raw_json parsing


In [18]:
print("=" * 60)
print("TEST 4.2 — Sleep (last 7 nights)")
print("=" * 60)
try:
    sleep_records = get_sleep(conn, days=7)
    print(f"✅ get_sleep(7): {len(sleep_records)} records")
    print(f"  {'Date':<12} {'Total':>7} {'Deep':>6} {'REM':>6} {'Quality'}")
    print("  " + "-" * 50)
    for s in sleep_records:
        print(
            f"  {s.date:<12} {str(s.total_sleep_hrs):>7} {str(s.deep_sleep_hrs):>6} "
            f"{str(s.rem_sleep_hrs):>6} {s.sleep_quality_label}"
        )
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.2 — Sleep (last 7 nights)
✅ get_sleep(7): 6 records
  Date           Total   Deep    REM Quality
  --------------------------------------------------
  2026-05-02      7.18    1.0   0.67 Adequate duration
  2026-05-01       7.6   1.63   1.22 Good duration
  2026-04-30      6.81   1.88   1.57 Adequate duration
  2026-04-29      6.88   1.07   0.97 Adequate duration
  2026-04-28      7.45   1.05   1.03 Adequate duration
  2026-04-27      7.43   1.28   1.35 Adequate duration


In [19]:
print("=" * 60)
print("TEST 4.3 — Body Battery (last 7 days)")
print("=" * 60)
try:
    bb_records = get_body_battery(conn, days=7)
    print(f"✅ get_body_battery(7): {len(bb_records)} records")
    print(f"  {'Date':<12} {'At Wake':>8} {'Highest':>8} {'Lowest':>8} {'Status'}")
    print("  " + "-" * 58)
    for b in bb_records:
        print(
            f"  {b.date:<12} {str(b.at_wake):>8} {str(b.highest):>8} "
            f"{str(b.lowest):>8} {b.wake_label}"
        )

    nulls = sum(1 for b in bb_records if b.at_wake is None)
    if nulls > 0:
        print(f"  ⚠️  {nulls} record(s) with null at_wake — check raw_json parsing")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.3 — Body Battery (last 7 days)
✅ get_body_battery(7): 7 records
  Date          At Wake  Highest   Lowest Status
  ----------------------------------------------------------
  2026-05-03         69       69       69 Adequately recovered
  2026-05-02         32       94       22 Partially recovered ⚠️
  2026-05-01         26       87       26 Poorly recovered ⚠️
  2026-04-30         41       89       22 Partially recovered ⚠️
  2026-04-29         52       99       30 Adequately recovered
  2026-04-28         70      100       43 Well recovered ✅
  2026-04-27         40      100       40 Partially recovered ⚠️


In [20]:
print("=" * 60)
print("TEST 4.4 — Stress (last 7 days)")
print("=" * 60)
try:
    stress_records = get_stress(conn, days=7)
    print(f"✅ get_stress(7): {len(stress_records)} records")
    print(f"  {'Date':<12} {'Avg':>6} {'Max':>6} {'Qualifier'}")
    print("  " + "-" * 45)
    for s in stress_records:
        print(
            f"  {s.date:<12} {str(s.avg_stress):>6} {str(s.max_stress):>6} "
            f"{s.stress_label}"
        )

    nulls = sum(1 for s in stress_records if s.avg_stress is None)
    if nulls > 0:
        print(f"  ⚠️  {nulls} record(s) with null avg_stress — check raw_json parsing")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.4 — Stress (last 7 days)
✅ get_stress(7): 7 records
  Date            Avg    Max Qualifier
  ---------------------------------------------
  2026-05-03       -1   None Unknown
  2026-05-02       29     98 Unknown
  2026-05-01       24     95 Unknown
  2026-04-30       21     91 Unknown
  2026-04-29       20     87 Unknown
  2026-04-28       18     90 Unknown
  2026-04-27       15     83 Unknown


In [21]:
print("=" * 60)
print("TEST 4.5 — Training Readiness (last 7 days)")
print("=" * 60)
try:
    ready_records = get_training_readiness(conn, days=7)
    print(f"✅ get_training_readiness(7): {len(ready_records)} records")
    print(f"  {'Date':<12} {'Score':>6} {'Level':<10} {'Feedback'}")
    print("  " + "-" * 55)
    for r in ready_records:
        print(
            f"  {r.date:<12} {str(r.score):>6} {str(r.level):<10} "
            f"{r.feedback_short or 'N/A'}"
        )
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.5 — Training Readiness (last 7 days)
✅ get_training_readiness(7): 7 records
  Date          Score Level      Feedback
  -------------------------------------------------------
  2026-05-03     68.0 MODERATE   GOOD_SLEEP_HISTORY
  2026-05-02     77.0 HIGH       WELL_RECOVERED
  2026-05-01     73.0 MODERATE   RECOVERED_AND_READY
  2026-04-30     68.0 MODERATE   RECOVERED_AND_READY
  2026-04-29     81.0 HIGH       WELL_RECOVERED
  2026-04-28     83.0 HIGH       WELL_RECOVERED
  2026-04-27     80.0 HIGH       WELL_RECOVERED


In [22]:
print("=" * 60)
print("TEST 4.6 — Resting HR (last 7 days)")
print("=" * 60)
try:
    hr_records = get_resting_hr(conn, days=7)
    print(f"✅ get_resting_hr(7): {len(hr_records)} records")
    print(f"  {'Date':<12} {'Resting HR':>11}")
    print("  " + "-" * 25)
    for h in hr_records:
        val = f"{h.resting_hr} bpm" if h.resting_hr else "N/A"
        print(f"  {h.date:<12} {val:>11}")
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.6 — Resting HR (last 7 days)
✅ get_resting_hr(7): 7 records
  Date          Resting HR
  -------------------------
  2026-05-03        85 bpm
  2026-05-02        41 bpm
  2026-05-01        39 bpm
  2026-04-30        40 bpm
  2026-04-29        38 bpm
  2026-04-28        42 bpm
  2026-04-27        41 bpm


In [23]:
print("=" * 60)
print("TEST 4.7 — Race Predictions")
print("=" * 60)
try:
    pred = get_race_predictions(conn)
    print(f"✅ get_race_predictions:")
    print(format_race_predictions(pred))
except Exception as e:
    print(f"❌ FAILED: {e}")

TEST 4.7 — Race Predictions
✅ get_race_predictions:
=== Garmin Race Predictions (2026-05-03) ===
  5K:             17:47
  10K:            37:47
  Half Marathon:  1:23:13
  Marathon:       3:01:12


In [24]:
print("=" * 60)
print("TEST 4.8 — Daily snapshots & recovery context (last 3 days)")
print("=" * 60)
try:
    snapshots = get_recent_snapshots(conn, days=3)
    print(f"✅ get_recent_snapshots(3): {len(snapshots)} snapshots")

    # Check composite recovery scores
    for s in snapshots:
        score = s.recovery_score
        score_str = f"{score:.0f}/100" if score is not None else "N/A"
        print(f"   {s.date} | recovery score: {score_str}")

    print("\n--- Agent-ready recovery context ---")
    print(format_recovery_context(snapshots, include_days=3))
except Exception as e:
    print(f"❌ FAILED: {e}")
finally:
    conn.close()
    print("\n✅ Garmin DB connection closed.")

TEST 4.8 — Daily snapshots & recovery context (last 3 days)
✅ get_recent_snapshots(3): 3 snapshots
   2026-05-03 | recovery score: 72/100
   2026-05-02 | recovery score: 64/100
   2026-05-01 | recovery score: 61/100

--- Agent-ready recovery context ---
=== Recovery Context ===

--- 2026-05-03 ---
  Recovery score (composite): 72/100
  Training readiness: 68.0 — Moderate
    Feedback: GOOD_SLEEP_HISTORY
    HRV factor: VERY_GOOD
    Sleep history: GOOD
  HRV: None (weekly avg: 100.0) — Balanced ✅, baseline N/A
  Body battery at wake: 69 — Adequately recovered (peak: 69, low: 69)
  Stress: avg -1, max None — Unknown
  Resting HR: 85 bpm

--- 2026-05-02 ---
  Recovery score (composite): 64/100
  Training readiness: 77.0 — Moderate
    Feedback: WELL_RECOVERED
    HRV factor: GOOD
    Sleep history: GOOD
  HRV: None (weekly avg: 97.0) — Balanced ✅, baseline N/A
  Sleep: 7.18h total (deep: 1.0h, REM: 0.67h) — Adequate duration
    Feedback: NEGATIVE_LONG_BUT_NOT_ENOUGH_REM
  Body battery a